In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/2025.csv", nrows=5)

df.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_Price,Max_Price,Modal_Price,Commodity_Code
0,Andhra Pradesh,Chittor,Chittoor,Gur (Jaggery),NO 2,FAQ,2025-01-01,3200.0,3500.0,3500.0,74
1,Tamil Nadu,The Nilgiris,Gudalur (Uzhavar Sandhai ),Garlic,Average,Local,2025-01-01,28000.0,30000.0,30000.0,25
2,Tamil Nadu,The Nilgiris,Gudalur (Uzhavar Sandhai ),Ginger (Green),Green Ginger,Local,2025-01-01,7000.0,8000.0,8000.0,103
3,Tamil Nadu,The Nilgiris,Gudalur (Uzhavar Sandhai ),Green Chilli,Green Chilly,Local,2025-01-01,4500.0,5000.0,5000.0,87
4,Tamil Nadu,The Nilgiris,Gudalur (Uzhavar Sandhai ),Mint (Pudina),Mint (Pudina),Local,2025-01-01,4500.0,5000.0,5000.0,360


In [2]:
df.shape

(5, 11)

In [3]:
df.columns

Index(['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade',
       'Arrival_Date', 'Min_Price', 'Max_Price', 'Modal_Price',
       'Commodity_Code'],
      dtype='str')

In [4]:
df["Commodity"].unique()

<ArrowStringArray>
['Gur (Jaggery)', 'Garlic', 'Ginger (Green)', 'Green Chilli', 'Mint (Pudina)']
Length: 5, dtype: str

In [5]:
df["Commodity"].value_counts()

Commodity
Gur (Jaggery)     1
Garlic            1
Ginger (Green)    1
Green Chilli      1
Mint (Pudina)     1
Name: count, dtype: int64

In [6]:
df[["Min_Price", "Max_Price", "Modal_Price"]].describe()

,Min_Price,Max_Price,Modal_Price
count,5.000000,5.000000,5.000000
mean,9440.000000,10300.000000,10300.000000
std,10466.279186,11133.283433,11133.283433
min,3200.000000,3500.000000,3500.000000
25%,4500.000000,5000.000000,5000.000000
50%,4500.000000,5000.000000,5000.000000
75%,7000.000000,8000.000000,8000.000000
max,28000.000000,30000.000000,30000.000000


In [7]:
df["Arrival_Date"].head()

0    2025-01-01
1    2025-01-01
2    2025-01-01
3    2025-01-01
4    2025-01-01
Name: Arrival_Date, dtype: str

In [8]:
df.dtypes

State                 str
District              str
Market                str
Commodity             str
Variety               str
Grade                 str
Arrival_Date          str
Min_Price         float64
Max_Price         float64
Modal_Price       float64
Commodity_Code      int64
dtype: object

In [9]:
total_rows = 0

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    total_rows += len(chunk)

total_rows

5819482

In [10]:
target_counts = {
    "Tomato": 0,
    "Onion": 0,
    "Potato": 0
}

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    for commodity in target_counts:
        target_counts[commodity] += (chunk["Commodity"] == commodity).sum()

target_counts

{'Tomato': np.int64(210960),
 'Onion': np.int64(220991),
 'Potato': np.int64(204136)}

In [11]:
date_min = None
date_max = None

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    dates = pd.to_datetime(chunk["Arrival_Date"], errors="coerce")

    chunk_min = dates.min()
    chunk_max = dates.max()

    if date_min is None or chunk_min < date_min:
        date_min = chunk_min

    if date_max is None or chunk_max > date_max:
        date_max = chunk_max

print("Earliest date:", date_min)
print("Latest date:", date_max)

Earliest date: 2025-01-01 00:00:00
Latest date: 2025-12-30 00:00:00


In [12]:
target_markets = {
    "Tomato": set(),
    "Onion": set(),
    "Potato": set()
}

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    for commodity in target_markets:
        markets = chunk.loc[
            chunk["Commodity"] == commodity,
            "Market"
        ].dropna().unique()

        target_markets[commodity].update(markets)

for commodity, markets in target_markets.items():
    print(commodity, ":", len(markets), "unique markets")

Tomato : 1914 unique markets
Onion : 1932 unique markets
Potato : 1857 unique markets


In [13]:
target_states = {
    "Tomato": set(),
    "Onion": set(),
    "Potato": set()
}

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    for commodity in target_states:
        states = chunk.loc[
            chunk["Commodity"] == commodity,
            "State"
        ].dropna().unique()

        target_states[commodity].update(states)

for commodity, states in target_states.items():
    print(commodity, ":", len(states), "unique states")
    

Tomato : 25 unique states
Onion : 28 unique states
Potato : 28 unique states


In [14]:
df_2024 = pd.read_csv("../data/raw/2024.csv", nrows=5)

df_2024.columns

Index(['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade',
       'Arrival_Date', 'Min_Price', 'Max_Price', 'Modal_Price',
       'Commodity_Code'],
      dtype='str')

In [15]:
date_min_2024 = None
date_max_2024 = None

for chunk in pd.read_csv("../data/raw/2024.csv", chunksize=100_000):
    dates = pd.to_datetime(chunk["Arrival_Date"], errors="coerce")

    chunk_min = dates.min()
    chunk_max = dates.max()

    if date_min_2024 is None or chunk_min < date_min_2024:
        date_min_2024 = chunk_min

    if date_max_2024 is None or chunk_max > date_max_2024:
        date_max_2024 = chunk_max

print("Earliest date:", date_min_2024)
print("Latest date:", date_max_2024)

Earliest date: 2024-01-01 00:00:00
Latest date: 2024-12-31 00:00:00


In [16]:
target_counts_2024 = {
    "Tomato": 0,
    "Onion": 0,
    "Potato": 0
}

for chunk in pd.read_csv("../data/raw/2024.csv", chunksize=100_000):
    for commodity in target_counts_2024:
        target_counts_2024[commodity] += (
            chunk["Commodity"] == commodity
        ).sum()

target_counts_2024

{'Tomato': np.int64(214864),
 'Onion': np.int64(225921),
 'Potato': np.int64(221842)}

In [17]:
target_markets_2024 = {
    "Tomato": set(),
    "Onion": set(),
    "Potato": set()
}

for chunk in pd.read_csv("../data/raw/2024.csv", chunksize=100_000):
    for commodity in target_markets_2024:
        markets = chunk.loc[
            chunk["Commodity"] == commodity,
            "Market"
        ].dropna().unique()

        target_markets_2024[commodity].update(markets)

for commodity, markets in target_markets_2024.items():
    print(commodity, ":", len(markets), "unique markets")

Tomato : 1254 unique markets
Onion : 1242 unique markets
Potato : 1182 unique markets


In [18]:
target_states_2024 = {
    "Tomato": set(),
    "Onion": set(),
    "Potato": set()
}

for chunk in pd.read_csv("../data/raw/2024.csv", chunksize=100_000):
    for commodity in target_states_2024:
        states = chunk.loc[
            chunk["Commodity"] == commodity,
            "State"
        ].dropna().unique()

        target_states_2024[commodity].update(states)

for commodity, states in target_states_2024.items():
    print(commodity, ":", len(states), "unique states")

Tomato : 25 unique states
Onion : 27 unique states
Potato : 28 unique states


In [19]:
tomato_markets_2024 = set()

for chunk in pd.read_csv("../data/raw/2024.csv", chunksize=100_000):
    markets = chunk.loc[
        chunk["Commodity"] == "Tomato",
        "Market"
    ].dropna().unique()

    tomato_markets_2024.update(markets)

tomato_markets_2025 = set()

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    markets = chunk.loc[
        chunk["Commodity"] == "Tomato",
        "Market"
    ].dropna().unique()

    tomato_markets_2025.update(markets)

common_tomato_markets = tomato_markets_2024.intersection(
    tomato_markets_2025
)

print("2024 Tomato markets:", len(tomato_markets_2024))
print("2025 Tomato markets:", len(tomato_markets_2025))
print("Common Tomato markets:", len(common_tomato_markets))

2024 Tomato markets: 1254
2025 Tomato markets: 1914
Common Tomato markets: 1075


In [20]:
onion_markets_2024 = set()

for chunk in pd.read_csv("../data/raw/2024.csv", chunksize=100_000):
    markets = chunk.loc[
        chunk["Commodity"] == "Onion",
        "Market"
    ].dropna().unique()

    onion_markets_2024.update(markets)

onion_markets_2025 = set()

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    markets = chunk.loc[
        chunk["Commodity"] == "Onion",
        "Market"
    ].dropna().unique()

    onion_markets_2025.update(markets)

common_onion_markets = onion_markets_2024.intersection(
    onion_markets_2025
)

print("2024 Onion markets:", len(onion_markets_2024))
print("2025 Onion markets:", len(onion_markets_2025))
print("Common Onion markets:", len(common_onion_markets))

2024 Onion markets: 1242
2025 Onion markets: 1932
Common Onion markets: 1073


In [21]:
potato_markets_2024 = set()

for chunk in pd.read_csv("../data/raw/2024.csv", chunksize=100_000):
    markets = chunk.loc[
        chunk["Commodity"] == "Potato",
        "Market"
    ].dropna().unique()

    potato_markets_2024.update(markets)


potato_markets_2025 = set()

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    markets = chunk.loc[
        chunk["Commodity"] == "Potato",
        "Market"
    ].dropna().unique()

    potato_markets_2025.update(markets)


common_potato_markets = potato_markets_2024.intersection(
    potato_markets_2025
)

print("2024 Potato markets:", len(potato_markets_2024))
print("2025 Potato markets:", len(potato_markets_2025))
print("Common Potato markets:", len(common_potato_markets))

2024 Potato markets: 1182
2025 Potato markets: 1857
Common Potato markets: 1025


In [22]:
for year in [2024, 2025]:

    print(f"\n--- {year} ---")

    for commodity in ["Tomato", "Onion", "Potato"]:

        dates = set()

        file_path = f"../data/raw/{year}.csv"

        for chunk in pd.read_csv(
            file_path,
            chunksize=100_000,
            usecols=["Commodity", "Arrival_Date"]
        ):
            commodity_dates = chunk.loc[
                chunk["Commodity"] == commodity,
                "Arrival_Date"
            ].dropna().unique()

            dates.update(commodity_dates)

        print(
            f"{commodity}: {len(dates)} unique dates "
            f"({min(dates)} to {max(dates)})"
        )


--- 2024 ---
Tomato: 366 unique dates (2024-01-01 to 2024-12-31)
Onion: 366 unique dates (2024-01-01 to 2024-12-31)
Potato: 366 unique dates (2024-01-01 to 2024-12-31)

--- 2025 ---
Tomato: 341 unique dates (2025-01-01 to 2025-12-30)
Onion: 341 unique dates (2025-01-01 to 2025-12-29)
Potato: 341 unique dates (2025-01-01 to 2025-12-29)


In [23]:
market_counts = {}

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market"]
):
    tomato = chunk[chunk["Commodity"] == "Tomato"]

    counts = tomato["Market"].value_counts()

    for market, count in counts.items():
        market_counts[market] = market_counts.get(market, 0) + count

tomato_market_counts = pd.Series(market_counts)

print("Number of Tomato markets:", len(tomato_market_counts))
print("\nObservations per market:")
print(tomato_market_counts.describe())

Number of Tomato markets: 1914

Observations per market:
count    1914.000000
mean      110.219436
std       118.175372
min         1.000000
25%         7.000000
50%        40.000000
75%       219.000000
max       558.000000
dtype: float64


In [24]:
market_counts = {}

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market"]
):
    onion = chunk[chunk["Commodity"] == "Onion"]

    counts = onion["Market"].value_counts()

    for market, count in counts.items():
        market_counts[market] = market_counts.get(market, 0) + count

onion_market_counts = pd.Series(market_counts)

print("Number of Onion markets:", len(onion_market_counts))
print("\nObservations per market:")
print(onion_market_counts.describe())

Number of Onion markets: 1932

Observations per market:
count    1932.000000
mean      114.384576
std       126.307077
min         1.000000
25%         7.000000
50%        43.000000
75%       230.000000
max       794.000000
dtype: float64


In [25]:
market_counts = {}

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market"]
):
    potato = chunk[chunk["Commodity"] == "Potato"]

    counts = potato["Market"].value_counts()

    for market, count in counts.items():
        market_counts[market] = market_counts.get(market, 0) + count

potato_market_counts = pd.Series(market_counts)

print("Number of Potato markets:", len(potato_market_counts))
print("\nObservations per market:")
print(potato_market_counts.describe())


Number of Potato markets: 1857

Observations per market:
count    1857.000000
mean      109.927841
std       118.439017
min         1.000000
25%         6.000000
50%        37.000000
75%       217.000000
max       559.000000
dtype: float64


In [26]:
top_tomato_market = tomato_market_counts.idxmax()

print("Market:", top_tomato_market)
print("Observations:", tomato_market_counts.max())



Market: Balugaon
Observations: 558


In [27]:
market_data = []

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=[
        "Commodity",
        "Market",
        "Arrival_Date",
        "Modal_Price"
    ]
):
    filtered = chunk[
        (chunk["Commodity"] == "Tomato") &
        (chunk["Market"] == top_tomato_market)
    ]

    market_data.append(filtered)

top_market_df = pd.concat(market_data, ignore_index=True)

top_market_df["Arrival_Date"] = pd.to_datetime(
    top_market_df["Arrival_Date"]
)

top_market_df = top_market_df.sort_values("Arrival_Date")

print(top_market_df.head())
print("\nDate range:")
print(top_market_df["Arrival_Date"].min(), "to", top_market_df["Arrival_Date"].max())

print("\nUnique dates:", top_market_df["Arrival_Date"].nunique())

     Market Commodity Arrival_Date  Modal_Price
0  Balugaon    Tomato   2025-01-01       1800.0
1  Balugaon    Tomato   2025-01-02       1800.0
2  Balugaon    Tomato   2025-01-06       1400.0
3  Balugaon    Tomato   2025-01-07       1400.0
4  Balugaon    Tomato   2025-01-08       1000.0

Date range:
2025-01-01 00:00:00 to 2025-11-04 00:00:00

Unique dates: 247


In [28]:
date_gaps = (
    top_market_df["Arrival_Date"]
    .drop_duplicates()
    .sort_values()
    .diff()
    .dt.days
    .dropna()
)

print("Date gap distribution:")
print(date_gaps.describe())

print("\nGap frequency:")
print(date_gaps.value_counts().sort_index())

Date gap distribution:
count    246.000000
mean       1.247967
std        0.755589
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        7.000000
Name: Arrival_Date, dtype: float64

Gap frequency:
Arrival_Date
1.0    211
2.0     21
3.0      7
4.0      5
6.0      1
7.0      1
Name: count, dtype: int64


In [29]:
thresholds = [30, 60, 90, 180]

print("Tomato 2025 market coverage:\n")

for threshold in thresholds:
    count = (tomato_market_counts >= threshold).sum()
    percentage = count / len(tomato_market_counts) * 100

    print(
        f"Markets with >= {threshold} observations: "
        f"{count} ({percentage:.2f}%)"
    )

Tomato 2025 market coverage:

Markets with >= 30 observations: 994 (51.93%)
Markets with >= 60 observations: 907 (47.39%)
Markets with >= 90 observations: 860 (44.93%)
Markets with >= 180 observations: 686 (35.84%)


In [30]:
thresholds = [30, 60, 90, 180]

print("Onion 2025 market coverage:\n")

for threshold in thresholds:
    count = (onion_market_counts >= threshold).sum()
    percentage = count / len(onion_market_counts) * 100

    print(
        f"Markets with >= {threshold} observations: "
        f"{count} ({percentage:.2f}%)"
    )

Onion 2025 market coverage:

Markets with >= 30 observations: 1000 (51.76%)
Markets with >= 60 observations: 924 (47.83%)
Markets with >= 90 observations: 867 (44.88%)
Markets with >= 180 observations: 680 (35.20%)


In [31]:
(onion_market_counts >= threshold).sum()

np.int64(680)

In [32]:
count / len(onion_market_counts) * 100

np.float64(35.196687370600415)

In [33]:
thresholds = [30, 60, 90, 180]

print("Potato 2025 market coverage:\n")

for threshold in thresholds:
    count = (potato_market_counts >= threshold).sum()
    percentage = count / len(potato_market_counts) * 100

    print(
        f"Markets with >= {threshold} observations: "
        f"{count} ({percentage:.2f}%)"
    )

Potato 2025 market coverage:

Markets with >= 30 observations: 950 (51.16%)
Markets with >= 60 observations: 886 (47.71%)
Markets with >= 90 observations: 841 (45.29%)
Markets with >= 180 observations: 684 (36.83%)


In [34]:
tomato_90_markets = tomato_market_counts[
    tomato_market_counts >= 90
].index

market_spans = []

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market", "Arrival_Date"]
):
    filtered = chunk[
        (chunk["Commodity"] == "Tomato") &
        (chunk["Market"].isin(tomato_90_markets))
    ].copy()

    filtered["Arrival_Date"] = pd.to_datetime(
        filtered["Arrival_Date"]
    )

    grouped = filtered.groupby("Market")["Arrival_Date"].agg(
        first_date="min",
        last_date="max"
    )

    market_spans.append(grouped)

tomato_market_spans = (
    pd.concat(market_spans)
    .groupby(level=0)
    .agg({
        "first_date": "min",
        "last_date": "max"
    })
)

tomato_market_spans["span_days"] = (
    tomato_market_spans["last_date"]
    - tomato_market_spans["first_date"]
).dt.days

print("Number of qualifying markets:", len(tomato_market_spans))
print("\nDate-span distribution:")
print(tomato_market_spans["span_days"].describe())

Number of qualifying markets: 860

Date-span distribution:
count    860.000000
mean     294.708140
std       28.260178
min      109.000000
25%      298.000000
50%      305.000000
75%      307.000000
max      361.000000
Name: span_days, dtype: float64


In [35]:
for threshold_days in [90, 180, 270]:

    count = (
        tomato_market_spans["span_days"] >= threshold_days
    ).sum()

    percentage = (
        count / len(tomato_market_spans) * 100
    )

    print(
        f"Markets with >= {threshold_days} days span: "
        f"{count} ({percentage:.2f}%)"
    )

Markets with >= 90 days span: 860 (100.00%)
Markets with >= 180 days span: 851 (98.95%)
Markets with >= 270 days span: 779 (90.58%)


In [36]:
onion_90_markets = onion_market_counts[
    onion_market_counts >= 90
].index

market_spans = []

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market", "Arrival_Date"]
):
    filtered = chunk[
        (chunk["Commodity"] == "Onion") &
        (chunk["Market"].isin(onion_90_markets))
    ].copy()

    filtered["Arrival_Date"] = pd.to_datetime(
        filtered["Arrival_Date"]
    )

    grouped = filtered.groupby("Market")["Arrival_Date"].agg(
        first_date="min",
        last_date="max"
    )

    market_spans.append(grouped)

onion_market_spans = (
    pd.concat(market_spans)
    .groupby(level=0)
    .agg({
        "first_date": "min",
        "last_date": "max"
    })
)

onion_market_spans["span_days"] = (
    onion_market_spans["last_date"]
    - onion_market_spans["first_date"]
).dt.days

print("Number of qualifying markets:", len(onion_market_spans))
print("\nDate-span distribution:")
print(onion_market_spans["span_days"].describe())

Number of qualifying markets: 867

Date-span distribution:
count    867.000000
mean     298.294118
std       23.369460
min      118.000000
25%      302.000000
50%      305.000000
75%      307.000000
max      361.000000
Name: span_days, dtype: float64


In [37]:
for threshold_days in [90, 180, 270]:

    count = (
        onion_market_spans["span_days"] >= threshold_days
    ).sum()

    percentage = (
        count / len(onion_market_spans) * 100
    )

    print(
        f"Markets with >= {threshold_days} days span: "
        f"{count} ({percentage:.2f}%)"
    )

Markets with >= 90 days span: 867 (100.00%)
Markets with >= 180 days span: 862 (99.42%)
Markets with >= 270 days span: 807 (93.08%)


In [38]:
potato_90_markets = potato_market_counts[
    potato_market_counts >= 90
].index

market_spans = []

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market", "Arrival_Date"]
):
    filtered = chunk[
        (chunk["Commodity"] == "Potato") &
        (chunk["Market"].isin(potato_90_markets))
    ].copy()

    filtered["Arrival_Date"] = pd.to_datetime(
        filtered["Arrival_Date"]
    )

    grouped = filtered.groupby("Market")["Arrival_Date"].agg(
        first_date="min",
        last_date="max"
    )

    market_spans.append(grouped)

potato_market_spans = (
    pd.concat(market_spans)
    .groupby(level=0)
    .agg({
        "first_date": "min",
        "last_date": "max"
    })
)

potato_market_spans["span_days"] = (
    potato_market_spans["last_date"]
    - potato_market_spans["first_date"]
).dt.days

print("Number of qualifying markets:", len(potato_market_spans))
print("\nDate-span distribution:")
print(potato_market_spans["span_days"].describe())

Number of qualifying markets: 841

Date-span distribution:
count    841.000000
mean     295.290131
std       28.633048
min       86.000000
25%      301.000000
50%      305.000000
75%      307.000000
max      361.000000
Name: span_days, dtype: float64


In [39]:

for threshold_days in [90, 180, 270]:

    count = (
        potato_market_spans["span_days"] >= threshold_days
    ).sum()

    percentage = (
        count / len(potato_market_spans) * 100
    )

    print(
        f"Markets with >= {threshold_days} days span: "
        f"{count} ({percentage:.2f}%)"
    )

Markets with >= 90 days span: 839 (99.76%)
Markets with >= 180 days span: 836 (99.41%)
Markets with >= 270 days span: 758 (90.13%)


In [40]:
tomato_90_2024 = set()

for chunk in pd.read_csv(
    "../data/raw/2024.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market"]
):
    filtered = chunk[
        chunk["Commodity"] == "Tomato"
    ]

    counts = filtered["Market"].value_counts()

    tomato_90_2024.update(
        counts[counts >= 90].index
    )


tomato_90_2025 = set(
    tomato_market_counts[
        tomato_market_counts >= 90
    ].index
)


common_tomato_90_markets = (
    tomato_90_2024.intersection(tomato_90_2025)
)

print(
    "2024 Tomato markets with >=90 observations:",
    len(tomato_90_2024)
)

print(
    "2025 Tomato markets with >=90 observations:",
    len(tomato_90_2025)
)

print(
    "Common markets with >=90 observations in both years:",
    len(common_tomato_90_markets)
)

2024 Tomato markets with >=90 observations: 0
2025 Tomato markets with >=90 observations: 860
Common markets with >=90 observations in both years: 0


In [42]:
counts = filtered["Market"].value_counts()

In [43]:
tomato_counts_2024 = {}

for chunk in pd.read_csv(
    "../data/raw/2024.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market"]
):
    filtered = chunk[
        chunk["Commodity"] == "Tomato"
    ]

    counts = filtered["Market"].value_counts()

    for market, count in counts.items():
        tomato_counts_2024[market] = (
            tomato_counts_2024.get(market, 0) + count
        )


tomato_market_counts_2024 = pd.Series(tomato_counts_2024)

tomato_90_2024 = set(
    tomato_market_counts_2024[
        tomato_market_counts_2024 >= 90
    ].index
)

tomato_90_2025 = set(
    tomato_market_counts[
        tomato_market_counts >= 90
    ].index
)

common_tomato_90_markets = (
    tomato_90_2024.intersection(
        tomato_90_2025
    )
)

print(
    "2024 Tomato markets with >=90 observations:",
    len(tomato_90_2024)
)

print(
    "2025 Tomato markets with >=90 observations:",
    len(tomato_90_2025)
)

print(
    "Common markets with >=90 observations in both years:",
    len(common_tomato_90_markets)
)

2024 Tomato markets with >=90 observations: 863
2025 Tomato markets with >=90 observations: 860
Common markets with >=90 observations in both years: 791


In [44]:
onion_counts_2024 = {}

for chunk in pd.read_csv(
    "../data/raw/2024.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market"]
):
    filtered = chunk[
        chunk["Commodity"] == "Onion"
    ]

    counts = filtered["Market"].value_counts()

    for market, count in counts.items():
        onion_counts_2024[market] = (
            onion_counts_2024.get(market, 0) + count
        )


onion_market_counts_2024 = pd.Series(onion_counts_2024)

onion_90_2024 = set(
    onion_market_counts_2024[
        onion_market_counts_2024 >= 90
    ].index
)

onion_90_2025 = set(
    onion_market_counts[
        onion_market_counts >= 90
    ].index
)

common_onion_90_markets = (
    onion_90_2024.intersection(
        onion_90_2025
    )
)

print(
    "2024 Onion markets with >=90 observations:",
    len(onion_90_2024)
)

print(
    "2025 Onion markets with >=90 observations:",
    len(onion_90_2025)
)

print(
    "Common markets with >=90 observations in both years:",
    len(common_onion_90_markets)
)

2024 Onion markets with >=90 observations: 884
2025 Onion markets with >=90 observations: 867
Common markets with >=90 observations in both years: 804


In [45]:
potato_counts_2024 = {}

for chunk in pd.read_csv(
    "../data/raw/2024.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market"]
):
    filtered = chunk[
        chunk["Commodity"] == "Potato"
    ]

    counts = filtered["Market"].value_counts()

    for market, count in counts.items():
        potato_counts_2024[market] = (
            potato_counts_2024.get(market, 0) + count
        )


potato_market_counts_2024 = pd.Series(potato_counts_2024)

potato_90_2024 = set(
    potato_market_counts_2024[
        potato_market_counts_2024 >= 90
    ].index
)

potato_90_2025 = set(
    potato_market_counts[
        potato_market_counts >= 90
    ].index
)

common_potato_90_markets = (
    potato_90_2024.intersection(
        potato_90_2025
    )
)

print(
    "2024 Potato markets with >=90 observations:",
    len(potato_90_2024)
)

print(
    "2025 Potato markets with >=90 observations:",
    len(potato_90_2025)
)

print(
    "Common markets with >=90 observations in both years:",
    len(common_potato_90_markets)
)

2024 Potato markets with >=90 observations: 868
2025 Potato markets with >=90 observations: 841
Common markets with >=90 observations in both years: 792


In [46]:
tomato_common_90_2024_2025 = common_tomato_90_markets

tomato_date_spans = {}

for chunk in pd.read_csv(
    "../data/raw/2024.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market", "Arrival_Date"]
):
    filtered = chunk[
        (chunk["Commodity"] == "Tomato") &
        (chunk["Market"].isin(tomato_common_90_2024_2025))
    ].copy()

    if not filtered.empty:
        filtered["Arrival_Date"] = pd.to_datetime(
            filtered["Arrival_Date"]
        )

        for market, group in filtered.groupby("Market"):
            min_date = group["Arrival_Date"].min()
            max_date = group["Arrival_Date"].max()

            if market not in tomato_date_spans:
                tomato_date_spans[market] = [min_date, max_date]
            else:
                tomato_date_spans[market][0] = min(
                    tomato_date_spans[market][0],
                    min_date
                )
                tomato_date_spans[market][1] = max(
                    tomato_date_spans[market][1],
                    max_date
                )

tomato_span_days = pd.Series({
    market: (dates[1] - dates[0]).days + 1
    for market, dates in tomato_date_spans.items()
})

print("Number of common Tomato markets:", len(tomato_span_days))

print("\nDate-span summary:")
print(tomato_span_days.describe())

print("\nMarkets with span >= 180 days:",
      (tomato_span_days >= 180).sum())

print("Markets with span >= 270 days:",
      (tomato_span_days >= 270).sum())

print("Markets with span >= 330 days:",
      (tomato_span_days >= 330).sum())

Number of common Tomato markets: 791

Date-span summary:
count    791.000000
mean     318.342604
std       80.329134
min       91.000000
25%      312.500000
50%      365.000000
75%      366.000000
max      366.000000
dtype: float64

Markets with span >= 180 days: 633
Markets with span >= 270 days: 608
Markets with span >= 330 days: 587


In [48]:
tomato_date_spans_2025 = {}

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market", "Arrival_Date"]
):
    filtered = chunk[
        (chunk["Commodity"] == "Tomato") &
        (chunk["Market"].isin(common_tomato_90_markets))
    ].copy()

    if not filtered.empty:
        filtered["Arrival_Date"] = pd.to_datetime(
            filtered["Arrival_Date"]
        )

        for market, group in filtered.groupby("Market"):
            min_date = group["Arrival_Date"].min()
            max_date = group["Arrival_Date"].max()

            if market not in tomato_date_spans_2025:
                tomato_date_spans_2025[market] = [
                    min_date,
                    max_date
                ]
            else:
                tomato_date_spans_2025[market][0] = min(
                    tomato_date_spans_2025[market][0],
                    min_date
                )

                tomato_date_spans_2025[market][1] = max(
                    tomato_date_spans_2025[market][1],
                    max_date
                )


tomato_span_days_2025 = pd.Series({
    market: (dates[1] - dates[0]).days + 1
    for market, dates in tomato_date_spans_2025.items()
})


print("2025 markets found:", len(tomato_date_spans_2025))

print(
    "2025 earliest date:",
    min(dates[0] for dates in tomato_date_spans_2025.values())
)

print(
    "2025 latest date:",
    max(dates[1] for dates in tomato_date_spans_2025.values())
)

print("\n2025 date-span summary:")
print(tomato_span_days_2025.describe())

print(
    "\nMarkets with span >= 180 days:",
    (tomato_span_days_2025 >= 180).sum()
)

print(
    "Markets with span >= 270 days:",
    (tomato_span_days_2025 >= 270).sum()
)

print(
    "Markets with span >= 330 days:",
    (tomato_span_days_2025 >= 330).sum()
)

2025 markets found: 791
2025 earliest date: 2025-01-01 00:00:00
2025 latest date: 2025-12-28 00:00:00

2025 date-span summary:
count    791.000000
mean     298.125158
std       24.483645
min      110.000000
25%      300.000000
50%      306.000000
75%      308.000000
max      362.000000
dtype: float64

Markets with span >= 180 days: 786
Markets with span >= 270 days: 738
Markets with span >= 330 days: 2


In [49]:
tomato_180_both_years = (
    (tomato_span_days >= 180) &
    (tomato_span_days_2025 >= 180)
)

tomato_270_both_years = (
    (tomato_span_days >= 270) &
    (tomato_span_days_2025 >= 270)
)

print(
    "Tomato markets with >=180 days in both years:",
    tomato_180_both_years.sum()
)

print(
    "Tomato markets with >=270 days in both years:",
    tomato_270_both_years.sum()
)

Tomato markets with >=180 days in both years: 628
Tomato markets with >=270 days in both years: 592


In [50]:
onion_date_spans_2024 = {}

for chunk in pd.read_csv(
    "../data/raw/2024.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market", "Arrival_Date"]
):
    filtered = chunk[
        (chunk["Commodity"] == "Onion") &
        (chunk["Market"].isin(common_onion_90_markets))
    ].copy()

    if not filtered.empty:
        filtered["Arrival_Date"] = pd.to_datetime(
            filtered["Arrival_Date"]
        )

        for market, group in filtered.groupby("Market"):
            min_date = group["Arrival_Date"].min()
            max_date = group["Arrival_Date"].max()

            if market not in onion_date_spans_2024:
                onion_date_spans_2024[market] = [
                    min_date,
                    max_date
                ]
            else:
                onion_date_spans_2024[market][0] = min(
                    onion_date_spans_2024[market][0],
                    min_date
                )

                onion_date_spans_2024[market][1] = max(
                    onion_date_spans_2024[market][1],
                    max_date
                )


onion_span_days_2024 = pd.Series({
    market: (dates[1] - dates[0]).days + 1
    for market, dates in onion_date_spans_2024.items()
})


print("2024 Onion markets found:",
      len(onion_span_days_2024))

print("\n2024 date-span summary:")
print(onion_span_days_2024.describe())

print(
    "\nMarkets with span >= 180 days:",
    (onion_span_days_2024 >= 180).sum()
)

print(
    "Markets with span >= 270 days:",
    (onion_span_days_2024 >= 270).sum()
)

2024 Onion markets found: 804

2024 date-span summary:
count    804.000000
mean     318.881841
std       79.456464
min       97.000000
25%      307.000000
50%      365.000000
75%      366.000000
max      366.000000
dtype: float64

Markets with span >= 180 days: 650
Markets with span >= 270 days: 619


In [51]:
onion_date_spans_2025 = {}

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market", "Arrival_Date"]
):
    filtered = chunk[
        (chunk["Commodity"] == "Onion") &
        (chunk["Market"].isin(common_onion_90_markets))
    ].copy()

    if not filtered.empty:
        filtered["Arrival_Date"] = pd.to_datetime(
            filtered["Arrival_Date"]
        )

        for market, group in filtered.groupby("Market"):
            min_date = group["Arrival_Date"].min()
            max_date = group["Arrival_Date"].max()

            if market not in onion_date_spans_2025:
                onion_date_spans_2025[market] = [
                    min_date,
                    max_date
                ]
            else:
                onion_date_spans_2025[market][0] = min(
                    onion_date_spans_2025[market][0],
                    min_date
                )

                onion_date_spans_2025[market][1] = max(
                    onion_date_spans_2025[market][1],
                    max_date
                )


onion_span_days_2025 = pd.Series({
    market: (dates[1] - dates[0]).days + 1
    for market, dates in onion_date_spans_2025.items()
})


print("2025 Onion markets found:",
      len(onion_span_days_2025))

print(
    "2025 earliest date:",
    min(dates[0] for dates in onion_date_spans_2025.values())
)

print(
    "2025 latest date:",
    max(dates[1] for dates in onion_date_spans_2025.values())
)

print("\n2025 date-span summary:")
print(onion_span_days_2025.describe())

print(
    "\nMarkets with span >= 180 days:",
    (onion_span_days_2025 >= 180).sum()
)

print(
    "Markets with span >= 270 days:",
    (onion_span_days_2025 >= 270).sum()
)

2025 Onion markets found: 804
2025 earliest date: 2025-01-01 00:00:00
2025 latest date: 2025-12-28 00:00:00

2025 date-span summary:
count    804.000000
mean     301.861940
std       18.329409
min      128.000000
25%      304.000000
50%      307.000000
75%      308.000000
max      362.000000
dtype: float64

Markets with span >= 180 days: 800
Markets with span >= 270 days: 771


In [52]:
onion_180_both_years = (
    (onion_span_days_2024 >= 180) &
    (onion_span_days_2025 >= 180)
)

onion_270_both_years = (
    (onion_span_days_2024 >= 270) &
    (onion_span_days_2025 >= 270)
)

print(
    "Onion markets with >=180 days in both years:",
    onion_180_both_years.sum()
)

print(
    "Onion markets with >=270 days in both years:",
    onion_270_both_years.sum()
)

Onion markets with >=180 days in both years: 646
Onion markets with >=270 days in both years: 595


In [53]:
potato_date_spans_2024 = {}

for chunk in pd.read_csv(
    "../data/raw/2024.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market", "Arrival_Date"]
):
    filtered = chunk[
        (chunk["Commodity"] == "Potato") &
        (chunk["Market"].isin(common_potato_90_markets))
    ].copy()

    if not filtered.empty:
        filtered["Arrival_Date"] = pd.to_datetime(
            filtered["Arrival_Date"]
        )

        for market, group in filtered.groupby("Market"):
            min_date = group["Arrival_Date"].min()
            max_date = group["Arrival_Date"].max()

            if market not in potato_date_spans_2024:
                potato_date_spans_2024[market] = [
                    min_date,
                    max_date
                ]
            else:
                potato_date_spans_2024[market][0] = min(
                    potato_date_spans_2024[market][0],
                    min_date
                )

                potato_date_spans_2024[market][1] = max(
                    potato_date_spans_2024[market][1],
                    max_date
                )


potato_span_days_2024 = pd.Series({
    market: (dates[1] - dates[0]).days + 1
    for market, dates in potato_date_spans_2024.items()
})


print(
    "2024 Potato markets found:",
    len(potato_span_days_2024)
)

print("\n2024 date-span summary:")
print(potato_span_days_2024.describe())

print(
    "\nMarkets with span >= 180 days:",
    (potato_span_days_2024 >= 180).sum()
)

print(
    "Markets with span >= 270 days:",
    (potato_span_days_2024 >= 270).sum()
)

2024 Potato markets found: 792

2024 date-span summary:
count    792.000000
mean     323.958333
std       76.463849
min      122.000000
25%      345.750000
50%      365.000000
75%      366.000000
max      366.000000
dtype: float64

Markets with span >= 180 days: 654
Markets with span >= 270 days: 630


In [54]:
potato_date_spans_2025 = {}

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=["Commodity", "Market", "Arrival_Date"]
):
    filtered = chunk[
        (chunk["Commodity"] == "Potato") &
        (chunk["Market"].isin(common_potato_90_markets))
    ].copy()

    if not filtered.empty:
        filtered["Arrival_Date"] = pd.to_datetime(
            filtered["Arrival_Date"]
        )

        for market, group in filtered.groupby("Market"):
            min_date = group["Arrival_Date"].min()
            max_date = group["Arrival_Date"].max()

            if market not in potato_date_spans_2025:
                potato_date_spans_2025[market] = [
                    min_date,
                    max_date
                ]
            else:
                potato_date_spans_2025[market][0] = min(
                    potato_date_spans_2025[market][0],
                    min_date
                )

                potato_date_spans_2025[market][1] = max(
                    potato_date_spans_2025[market][1],
                    max_date
                )


potato_span_days_2025 = pd.Series({
    market: (dates[1] - dates[0]).days + 1
    for market, dates in potato_date_spans_2025.items()
})


print(
    "2025 Potato markets found:",
    len(potato_span_days_2025)
)

print(
    "2025 earliest date:",
    min(dates[0] for dates in potato_date_spans_2025.values())
)

print(
    "2025 latest date:",
    max(dates[1] for dates in potato_date_spans_2025.values())
)

print("\n2025 date-span summary:")
print(potato_span_days_2025.describe())

print(
    "\nMarkets with span >= 180 days:",
    (potato_span_days_2025 >= 180).sum()
)

print(
    "Markets with span >= 270 days:",
    (potato_span_days_2025 >= 270).sum()
)


2025 Potato markets found: 792
2025 earliest date: 2025-01-01 00:00:00
2025 latest date: 2025-12-28 00:00:00

2025 date-span summary:
count    792.000000
mean     297.935606
std       26.630632
min       87.000000
25%      302.000000
50%      307.000000
75%      308.000000
max      362.000000
dtype: float64

Markets with span >= 180 days: 788
Markets with span >= 270 days: 729


In [55]:
potato_180_both_years = (
    (potato_span_days_2024 >= 180) &
    (potato_span_days_2025 >= 180)
)

potato_270_both_years = (
    (potato_span_days_2024 >= 270) &
    (potato_span_days_2025 >= 270)
)

print(
    "Potato markets with >=180 days in both years:",
    potato_180_both_years.sum()
)

print(
    "Potato markets with >=270 days in both years:",
    potato_270_both_years.sum()
)

Potato markets with >=180 days in both years: 650
Potato markets with >=270 days in both years: 608


In [1]:
import pandas as pd

file_path = "../data/raw/2025.csv"

total_rows = 0
missing_counts = {}
duplicate_count = 0

for chunk in pd.read_csv(
    file_path,
    chunksize=100_000
):
    total_rows += len(chunk)

    # Missing values
    chunk_missing = chunk.isna().sum()


    for column, count in chunk_missing.items():
        missing_counts[column] = (
            missing_counts.get(column, 0) + count
        )

    # Exact duplicate rows
    duplicate_count += chunk.duplicated().sum()


print("Total rows:", total_rows)

print("\nMissing values:")
print(pd.Series(missing_counts))

print("\nDuplicate rows:", duplicate_count)

Total rows: 5819482

Missing values:
State             0
District          0
Market            0
Commodity         0
Variety           0
Grade             0
Arrival_Date      0
Min_Price         0
Max_Price         0
Modal_Price       0
Commodity_Code    0
dtype: int64

Duplicate rows: 3081


In [2]:
sample = pd.read_csv(
    "../data/raw/2025.csv",
    nrows=100_000
)

duplicates = sample[sample.duplicated(keep=False)]

print("Duplicate rows in first 100,000 rows:", len(duplicates))

duplicates.head(10)

Duplicate rows in first 100,000 rows: 0


,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_Price,Max_Price,Modal_Price,Commodity_Code


In [3]:
cleaned_rows = 0
duplicate_rows = 0

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000
):
    before = len(chunk)

    chunk = chunk.drop_duplicates()

    after = len(chunk)

    cleaned_rows += after
    duplicate_rows += before - after


print("Original rows:", 5819482)
print("Duplicate rows removed:", duplicate_rows)
print("Rows after duplicate removal:", cleaned_rows)

Original rows: 5819482
Duplicate rows removed: 3081
Rows after duplicate removal: 5816401


In [4]:
invalid_negative = 0
invalid_zero = 0
invalid_min_max = 0
invalid_modal_range = 0

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=[
        "Min_Price",
        "Max_Price",
        "Modal_Price"
    ]
):
    invalid_negative += (
        (chunk[
            ["Min_Price", "Max_Price", "Modal_Price"]
        ] < 0).any(axis=1)
    ).sum()

    invalid_zero += (
        (chunk[
            ["Min_Price", "Max_Price", "Modal_Price"]
        ] == 0).any(axis=1)
    ).sum()

    invalid_min_max += (
        chunk["Min_Price"] > chunk["Max_Price"]
    ).sum()

    invalid_modal_range += (
        (chunk["Modal_Price"] < chunk["Min_Price"]) |
        (chunk["Modal_Price"] > chunk["Max_Price"])
    ).sum()


print("Rows with any negative price:", invalid_negative)
print("Rows with any zero price:", invalid_zero)
print("Rows with Min_Price > Max_Price:", invalid_min_max)
print(
    "Rows with Modal_Price outside Min-Max range:",
    invalid_modal_range
)

Rows with any negative price: 0
Rows with any zero price: 5315
Rows with Min_Price > Max_Price: 897
Rows with Modal_Price outside Min-Max range: 4720


In [5]:
for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=[
        "State",
        "Market",
        "Commodity",
        "Arrival_Date",
        "Min_Price",
        "Max_Price",
        "Modal_Price"
    ]
):
    invalid = chunk[
        (chunk["Min_Price"] == 0) |
        (chunk["Max_Price"] == 0) |
        (chunk["Modal_Price"] == 0) |
        (chunk["Min_Price"] > chunk["Max_Price"]) |
        (chunk["Modal_Price"] < chunk["Min_Price"]) |
        (chunk["Modal_Price"] > chunk["Max_Price"])
    ]

    if not invalid.empty:
        display(invalid.head(10))
        break

,State,Market,Commodity,Arrival_Date,Min_Price,Max_Price,Modal_Price
302,Tamil Nadu,Theni (Uzhavar Sandhai ),Chow Chow,2025-01-01,0.0,2400.0,2400.0
303,Tamil Nadu,Theni (Uzhavar Sandhai ),Cluster beans,2025-01-01,0.0,4000.0,4000.0
304,Tamil Nadu,Theni (Uzhavar Sandhai ),Coconut,2025-01-01,0.0,6000.0,6000.0
305,Tamil Nadu,Theni (Uzhavar Sandhai ),Colacasia,2025-01-01,0.0,4500.0,4500.0
306,Tamil Nadu,Theni (Uzhavar Sandhai ),Coriander (Leaves),2025-01-01,0.0,3200.0,3200.0
307,Tamil Nadu,Theni (Uzhavar Sandhai ),Cowpea (Veg),2025-01-01,0.0,4500.0,4500.0
308,Tamil Nadu,Theni (Uzhavar Sandhai ),Elephant Yam (Suran),2025-01-01,0.0,4500.0,4500.0
310,Tamil Nadu,Theni (Uzhavar Sandhai ),Ginger (Green),2025-01-01,0.0,4500.0,4500.0
311,Tamil Nadu,Theni (Uzhavar Sandhai ),Grapes,2025-01-01,0.0,8000.0,8000.0
314,Tamil Nadu,Theni (Uzhavar Sandhai ),Green Peas,2025-01-01,0.0,10000.0,10000.0


In [6]:
target_commodities = ["Tomato", "Onion", "Potato"]

target_rows = 0
target_zero_price = 0
target_min_max_invalid = 0
target_modal_invalid = 0

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000,
    usecols=[
        "Commodity",
        "Min_Price",
        "Max_Price",
        "Modal_Price"
    ]
):
    filtered = chunk[
        chunk["Commodity"].isin(target_commodities)
    ]

    target_rows += len(filtered)

    target_zero_price += (
        (
            filtered[
                ["Min_Price", "Max_Price", "Modal_Price"]
            ] == 0
        ).any(axis=1)
    ).sum()

    target_min_max_invalid += (
        filtered["Min_Price"] >
        filtered["Max_Price"]
    ).sum()

    target_modal_invalid += (
        (filtered["Modal_Price"] < filtered["Min_Price"]) |
        (filtered["Modal_Price"] > filtered["Max_Price"])
    ).sum()


print("Target commodity rows:", target_rows)
print("Rows with any zero price:", target_zero_price)
print("Rows with Min_Price > Max_Price:", target_min_max_invalid)
print(
    "Rows with Modal_Price outside Min-Max:",
    target_modal_invalid
)

Target commodity rows: 636087
Rows with any zero price: 98
Rows with Min_Price > Max_Price: 59
Rows with Modal_Price outside Min-Max: 68


In [7]:
target_commodities = ["Tomato", "Onion", "Potato"]

cleaned_chunks = []

for chunk in pd.read_csv(
    "../data/raw/2025.csv",
    chunksize=100_000
):
    # Keep target commodities only
    chunk = chunk[
        chunk["Commodity"].isin(target_commodities)
    ].copy()

    # Remove exact duplicates
    chunk = chunk.drop_duplicates()

    # Remove invalid price rows
    invalid_price = (
        (chunk["Min_Price"] < 0) |
        (chunk["Max_Price"] < 0) |
        (chunk["Modal_Price"] < 0) |
        (chunk["Min_Price"] > chunk["Max_Price"]) |
        (chunk["Modal_Price"] < chunk["Min_Price"]) |
        (chunk["Modal_Price"] > chunk["Max_Price"]) |
        (chunk["Min_Price"] == 0) |
        (chunk["Max_Price"] == 0) |
        (chunk["Modal_Price"] == 0)
    )

    chunk = chunk[~invalid_price]

    cleaned_chunks.append(chunk)


cleaned_2025 = pd.concat(
    cleaned_chunks,
    ignore_index=True
)

print("Cleaned target rows:", len(cleaned_2025))

print("\nCommodity counts:")
print(cleaned_2025["Commodity"].value_counts())

print("\nRemaining missing values:")
print(cleaned_2025.isna().sum())

print("\nRemaining duplicate rows:",
      cleaned_2025.duplicated().sum())

Cleaned target rows: 635761

Commodity counts:
Commodity
Onion     220857
Tomato    210860
Potato    204044
Name: count, dtype: int64

Remaining missing values:
State             0
District          0
Market            0
Commodity         0
Variety           0
Grade             0
Arrival_Date      0
Min_Price         0
Max_Price         0
Modal_Price       0
Commodity_Code    0
dtype: int64

Remaining duplicate rows: 0


In [8]:
cleaned_2025.to_csv(
    "../data/processed/cleaned_2025.csv",
    index=False
)

print("Saved cleaned dataset.")

Saved cleaned dataset.
